# 05. Neural evaluator

`NeuralEvaluator` wraps a trained (or untrained) `AllenFormulaNet` as an MCTS
`Evaluator`. One inference pass per `evaluate(node)` returns:

* `priors`: a probability over the productions that are legal at the node (illegal
  logits masked to `-inf` before the softmax), which `PUCTSelection` reads to guide
  the search.
* `value`: the network's value-head output, optionally scaled to reward units.

This is the bridge between the network (component 04) and the search (component
03): pass a `NeuralEvaluator` as `run_self_play`'s `network_evaluator` and the
search becomes AlphaZero-guided. It needs the `[nn]` extra (`torch`).

In [1]:
import alpha_rule
import torch
print("alpha_rule", alpha_rule.__version__, "| torch", torch.__version__)

alpha_rule 0.1.0 | torch 2.11.0+cu128


## Elements

`alpha_rule.evaluation`
* `NeuralEvaluator(model, grammar, *, max_len, value_scale=1.0)`: an `Evaluator`.
  `evaluate(node)` returns `EvalResult(value, priors)`.
  * `priors`: softmax over only the grammar-applicable productions at `node`.
  * `value`: the model's tanh value output times `value_scale` (default `1.0`, raw
    passthrough; set it to the training-time `value_scale` to recover reward units).
* It uses `model.predict` (gradient-free, restores train/eval mode), so scoring
  nodes mid-training never leaves the model stuck in eval mode.

## Scoring a node

`evaluate` reads only `node.name` and the grammar's applicable productions. The
priors cover exactly the legal actions and sum to 1; at the root, `END_RULE` is not
legal, so it does not appear.

In [2]:
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.nn import GrammarTokenizer, AllenFormulaNet
from alpha_rule.evaluation import NeuralEvaluator

torch.manual_seed(0)
g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
tok = GrammarTokenizer(g)
model = AllenFormulaNet(tok, d_model=16, nhead=2, num_layers=1, max_len=16)
ev = NeuralEvaluator(model, g, max_len=16)

root = g.root()
res = ev.evaluate(root)
print("applicable at root :", [p.name for p in g.applicable_productions(root)])
print("priors             :", {k: round(v, 3) for k, v in res.priors.items()})
print("priors sum to      :", round(sum(res.priors.values()), 6))
print("value (scale 1.0)  :", round(res.value, 4))
print("END_RULE in priors :", "END_RULE" in res.priors)

applicable at root : ['A', 'B']
priors             : {'A': 0.461, 'B': 0.539}
priors sum to      : 1.0
value (scale 1.0)  : -0.4284
END_RULE in priors : False


## Priors guide the search

Pass the evaluator as `run_self_play`'s `network_evaluator` and PUCT uses its priors
on every expanded node, while the `simulator` supplies the leaf rewards. (Here the
simulator is a trivial constant; in practice it is the RL `RuleSimulator` from
component 07.)

In [3]:
import numpy as np
from alpha_rule.mcts import run_self_play

class ConstantSimulator:
    reward_scale = 1.0
    def evaluate(self, node):
        return 0.5

traj = run_self_play(
    grammar=g,
    simulator=ConstantSimulator(),
    network_evaluator=ev,            # priors come from the network
    n_simulations=20,
    depth_limit=3,
    rng=np.random.default_rng(0),
)
print("trajectory steps:", len(traj.steps))
for s in traj.steps:
    print(f"  {s.state!r:10} chose {s.next_state!r}")

trajectory steps: 2
  '<ROOT>'   chose 'B'
  'B'        chose 'B <END>'


## Training shifts the priors

The evaluator holds a live reference to the model, so training the model changes
what `evaluate` returns. Here we nudge the policy at the root toward `A` and the
value upward, then re-score the root. `evaluate` uses `model.predict`, which
restores train mode, so interleaving scoring and training is safe.

In [4]:
from alpha_rule.nn import train_step, default_optimizer

before = ev.evaluate(root)
optim = default_optimizer(model, lr=5e-3)
for _ in range(150):
    train_step(model, optim, [("<ROOT>", {"A": 1.0}, 0.8)], max_len=16)
after = ev.evaluate(root)

print(f"prior['A'] before {before.priors['A']:.3f}  after {after.priors['A']:.3f}")
print(f"prior['B'] before {before.priors['B']:.3f}  after {after.priors['B']:.3f}")
print(f"value      before {before.value:+.3f}  after {after.value:+.3f}")

assert after.priors["A"] > before.priors["A"] and after.priors["A"] > after.priors["B"]
print("training pushed the root prior toward A (ok)")

prior['A'] before 0.461  after 0.999
prior['B'] before 0.539  after 0.001
value      before -0.428  after +0.800
training pushed the root prior toward A (ok)


## value_scale

`value_scale` multiplies the network's `(-1, +1)` output to recover reward units.
Pass the same `value_scale` you trained with; the default `1.0` returns the raw
network value.

In [5]:
ev_raw = NeuralEvaluator(model, g, max_len=16, value_scale=1.0)
ev_scaled = NeuralEvaluator(model, g, max_len=16, value_scale=5.0)
print("value at scale 1.0:", round(ev_raw.evaluate(root).value, 4))
print("value at scale 5.0:", round(ev_scaled.evaluate(root).value, 4), "(5x the raw value)")

value at scale 1.0: 0.8005
value at scale 5.0: 4.0023 (5x the raw value)


## Basic checks

Asserts mirroring the unit tests. `ok` means the evaluator behaves.

In [6]:
import torch
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.nn import GrammarTokenizer, AllenFormulaNet
from alpha_rule.evaluation import NeuralEvaluator, Evaluator

torch.manual_seed(0)
g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<",))
tok = GrammarTokenizer(g)
model = AllenFormulaNet(tok, d_model=16, nhead=2, num_layers=1, max_len=16)
ev = NeuralEvaluator(model, g, max_len=16)
root = g.root()
res = ev.evaluate(root)

# priors cover exactly the applicable productions, sum to 1, finite value.
assert set(res.priors) == {p.name for p in g.applicable_productions(root)}
assert "END_RULE" not in res.priors
assert abs(sum(res.priors.values()) - 1.0) < 1e-5
assert res.value == res.value                         # not NaN

# It is an Evaluator, is deterministic, and leaves the model's mode intact.
assert isinstance(ev, Evaluator)
assert model.training is True                         # predict restored train mode
a = ev.evaluate(root)
b = ev.evaluate(root)
assert abs(a.value - b.value) < 1e-6

# value_scale must be positive.
try:
    NeuralEvaluator(model, g, max_len=16, value_scale=0.0)
    raise AssertionError("expected ValueError")
except ValueError:
    pass

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_neural_evaluator
```

Needs the `[nn]` extra (`torch`); without it `run_tests` skips the module.